# WAXAL-NET zero-shot: whisper_large_v3

Evaluates on the WAXAL test split for: acholi, luganda, soga, masaaba, nyankole, akan, lingala, dagbani, ewe, shona, ikposo, dagaare, malagasy, fula, tigrinya, amharic, oromo, sidama, wolaytta.

In [ ]:
MODEL = "whisper_large_v3"
LANGUAGES = ['acholi', 'luganda', 'soga', 'masaaba', 'nyankole', 'akan', 'lingala', 'dagbani', 'ewe', 'shona', 'ikposo', 'dagaare', 'malagasy', 'fula', 'tigrinya', 'amharic', 'oromo', 'sidama', 'wolaytta']

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

SESSION_START = time.time()


def find_repo():
    for depth in range(1, 5):
        for hit in Path("/kaggle/input").glob("*/" * depth + "src/train.py"):
            return hit.parent.parent
    raise FileNotFoundError("Code not found: attach the waxal-asr-code dataset (Add Input).")


REPO = find_repo()
PY = "/tmp/waxal-venv/bin/python"
os.environ.update(HF_HOME="/tmp/hf_cache", HF_DATASETS_DISABLE_PROGRESS_BARS="1", WANDB_DISABLED="true",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
TOKEN_FILES = sorted(Path("/kaggle/input").glob("**/hf_token.txt"))
if TOKEN_FILES:
    os.environ["HF_TOKEN"] = TOKEN_FILES[0].read_text().strip()
else:
    print("No hf_token.txt input (waxal-hf-token dataset); checkpoints will not survive this session.")


def run(*command, check=True):
    """Run a command from the repo root, streaming its output into the notebook log."""
    process = subprocess.Popen(list(command), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() and check:
        raise RuntimeError(f"{' '.join(map(str, command))} exited with {process.returncode}")
    return process.returncode


print("Code:", REPO)

In [ ]:
run("bash", f"{REPO}/kaggle/setup_env.sh")

In [ ]:
failed = {}
for language in LANGUAGES:
    if time.time() > SESSION_START + 11 * 3600:
        failed[language] = "skipped: session time"
        continue
    print(f"===== {language} =====", flush=True)
    if run(PY, "-m", "src.zero_shot_eval", "--config", f"configs/experiment/zero_shot_{language}_{MODEL}.yaml", check=False):
        failed[language] = "failed (see log above)"

import json
for language in LANGUAGES:
    path = Path(f"/kaggle/working/outputs/zero_shot_{language}_{MODEL}/metrics.json")
    metrics = json.loads(path.read_text()) if path.exists() else {}
    status = metrics.get("status") or (f"WER {metrics['wer_percent']:.1f}% | CER {metrics['cer_percent']:.1f}%" if metrics else failed.get(language))
    print(f"{language}: {status}")